# CFO Quickstart: Constrained Flow Optimization

This notebook demonstrates how to run **Constrained Flow Optimization (CFO)** with FlowMol and Adjoint Matching in a few lines of code.

**CFO** (Algorithm 1 in the paper) solves:

$$\arg\max_\pi \; \mathbb{E}_{x \sim p^\pi_1} [r(x)] - \alpha D_{KL}(p^\pi_1 \| p^{\text{pre}}_1) \quad \text{s.t.} \quad \mathbb{E}_{x \sim p^\pi_1} [c(x)] \leq B$$

by reducing it to a sequence of unconstrained fine-tuning subproblems via an augmented Lagrangian scheme.

**Three pluggable components:**
1. **CFO** (`src/cfo/`) — the outer ALM algorithm
2. **FineTuningSolver** (`src/finetuning_solver/`) — Adjoint Matching (inner solver)
3. **Model** (`flowmol/`) — pretrained flow-based generative model

In [ ]:
import sys, os
from pathlib import Path

# Set working directory to repo root
os.chdir(Path(__file__).resolve().parent.parent if "__file__" in dir() else Path.cwd().parent)
sys.path.insert(0, str(Path.cwd() / "src"))

## 1. Imports and Configuration

In [ ]:
import copy
import os.path as osp
import torch
import torch.nn as nn
from omegaconf import OmegaConf

import flowmol
from cfo import AugmentedLagrangian, AugmentedReward, run_cfo, set_seed
from finetuning_solver.adjoint_matching import AdjointMatchingFinetuningTrainerFlowMol
from regressor import GNN, EGNN
from utils.sampling import sampling

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
# Load config (edit configs/cfo.yaml or override below)
config = OmegaConf.load("configs/cfo.yaml")

# Override for a quick demo run
config.seed = 42
config.total_steps = 4                             # total AM steps
config.augmented_lagrangian.lagrangian_updates = 2  # K = 2 ALM rounds
config.reward_lambda = 50.0
config.adjoint_matching.sampling.num_samples = 4
config.adjoint_matching.batch_size = 2
config.reward_sampling.num_samples = 8
config.augmented_lagrangian.sampling.num_samples = 8

set_seed(config.seed)

## 2. Load Models

Load the pretrained FlowMol generative model and GNN/EGNN property predictors for reward and constraint.

In [ ]:
def load_regressor(rc_config, device):
    """Load a pretrained GNN or EGNN property predictor."""
    K_x, K_a, K_c, K_e = 3, 10, 6, 5
    model_path = osp.join("pretrained_models", str(rc_config.fn), str(rc_config.model_type), str(rc_config.date), "best_model.pt")
    state = torch.load(model_path, map_location=device)
    if rc_config.model_type == "gnn":
        model = GNN(property=state["config"]["property"], node_feats=K_a+K_c+K_x, edge_feats=K_e,
                     hidden_dim=state["config"]["hidden_dim"], depth=state["config"]["depth"])
    elif rc_config.model_type == "egnn":
        model = EGNN(property=state["config"]["property"], num_atom_types=K_a, num_charge_classes=K_c,
                      num_bond_types=K_e, hidden_dim=state["config"]["hidden_dim"], depth=state["config"]["depth"])
    model.load_state_dict(state["model_state"])
    return model

# Load FlowMol
base_model = flowmol.load_pretrained(config.flow_model)
base_model.to(device)
fine_model = copy.deepcopy(base_model)

# Load reward and constraint models
reward_model = load_regressor(config.reward, device)
constraint_model = load_regressor(config.constraint, device)

print(f"Reward: {config.reward.fn} ({config.reward.model_type})")
print(f"Constraint: {config.constraint.fn} <= {config.constraint.bound} ({config.constraint.model_type})")

## 3. Wire Up CFO Components

Create the three pluggable pieces and pass them to `run_cfo()`.

In [ ]:
reward_lambda = config.reward_lambda
config.adjoint_matching.reward_lambda = reward_lambda
n_atoms = config.get("n_atoms", None)
min_num_atoms = config.get("min_num_atoms", None)
max_num_atoms = config.get("max_num_atoms", None)
config.adjoint_matching.sampling.n_atoms = n_atoms
config.adjoint_matching.sampling.min_num_atoms = min_num_atoms
config.adjoint_matching.sampling.max_num_atoms = max_num_atoms

# 1. AugmentedReward: computes f_k(x) = r(x) - penalty(c(x))
augmented_reward = AugmentedReward(
    reward_fn=reward_model,
    constraint_fn=constraint_model,
    alpha=reward_lambda,
    bound=config.constraint.bound,
    device=device,
)
augmented_reward.set_lambda_rho(
    lambda_=config.augmented_lagrangian.lambda_init,
    rho_=config.augmented_lagrangian.rho_init,
)

# 2. AugmentedLagrangian: manages lambda/rho updates (Steps 3-5)
alm = AugmentedLagrangian(
    config=config.augmented_lagrangian,
    constraint_fn=constraint_model,
    bound=config.constraint.bound,
    device=device,
)

# 3. FineTuningSolver factory: creates Adjoint Matching trainer each ALM round
def create_trainer_fn(config, model, base_model, grad_reward_fn, device, verbose):
    return AdjointMatchingFinetuningTrainerFlowMol(
        config=config, model=model, base_model=base_model,
        grad_reward_fn=grad_reward_fn, device=device, verbose=verbose,
    )

# 4. Sampling function (FlowMol-specific)
def sampling_fn(model):
    return sampling(config.reward_sampling, model, device=device,
                    n_atoms=n_atoms, min_num_atoms=min_num_atoms, max_num_atoms=max_num_atoms)

## 4. Run CFO

One function call runs the full Algorithm 1 loop.

In [ ]:
lagrangian_updates = config.augmented_lagrangian.lagrangian_updates  # K
num_iterations = config.total_steps // lagrangian_updates             # N

fine_model, full_stats, al_stats, models_list = run_cfo(
    config=config,
    base_model=base_model,
    fine_model=fine_model,
    augmented_reward=augmented_reward,
    alm=alm,
    create_trainer_fn=create_trainer_fn,
    sampling_fn=sampling_fn,
    device=device,
    num_iterations=num_iterations,
    lagrangian_updates=lagrangian_updates,
    plotting_freq=1,
    verbose=True,
)

## 5. Results

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.DataFrame.from_records(full_stats)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].plot(df["reward"], marker="o")
axes[0].set_title("Reward")
axes[0].set_xlabel("Step")

axes[1].plot(df["constraint"], marker="o", color="tab:orange")
axes[1].axhline(y=config.constraint.bound, color="red", linestyle="--", label=f"Bound={config.constraint.bound}")
axes[1].set_title("Constraint")
axes[1].set_xlabel("Step")
axes[1].legend()

axes[2].plot(df["constraint_violations"], marker="o", color="tab:green")
axes[2].set_title("Constraint Violations")
axes[2].set_xlabel("Step")

plt.tight_layout()
plt.show()

print(f"\nFinal reward: {full_stats[-1]['reward']:.4f}")
print(f"Final constraint: {full_stats[-1]['constraint']:.4f} (bound: {config.constraint.bound})")
print(f"Final violations: {full_stats[-1]['constraint_violations']:.4f}")

## Adapting CFO to Another Model

To use CFO with a different diffusion/flow model, you only need to replace two functions:

1. **`create_trainer_fn`** — Return a trainer object with `.generate_dataset()` and `.finetune()` methods
2. **`sampling_fn`** — Return `(dgl_mols_list, rd_mols_list)` from your model

Everything in `src/cfo/` (ALM, AugmentedReward, runner) stays unchanged.